# Análise inferencial — Violência contra as mulheres no RS

Camada 2 da pipeline: testes estatísticos sobre as tabelas produzidas pela Camada 1
(`outputs/tables/violencia_mensal_municipio.csv`, `violencia_anual_municipio.csv`,
`violencia_anual_municipio_taxa.csv`). A lógica de cada teste vive em `src/analysis/*.py`
— este notebook só chama essas funções e visualiza os resultados, sem reimplementar
estatística aqui.

Quatro perguntas, uma por seção:

1. **Tendência** (`tendencia.py`) — a série anual 2018-2025 tem tendência significativa, por tipo de crime?
2. **Sazonalidade** (`sazonalidade.py`) — os meses do ano diferem entre si, por tipo de crime?
3. **Correlação** (`correlacao.py`) — os 5 tipos de crime se correlacionam entre municípios?
4. **Quebra estrutural** (`quebra_estrutural.py`) — há uma quebra visível em torno das enchentes de maio/2024?


In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.analysis import correlacao, quebra_estrutural, sazonalidade, tendencia

# Mesma ordem e paleta fixas do notebook 1 (analise_exploratoria.ipynb) -- nunca
# recicladas por ranking/filtro, para manter a identidade visual de cada
# categoria consistente entre os dois notebooks.
CRIME_ORDER = [
    "Ameaça",
    "Estupro",
    "Feminicídio Consumado",
    "Feminicídio Tentado",
    "Lesão Corporal",
]
CRIME_COLORS = {
    "Ameaça": "#2a78d6",
    "Estupro": "#1baf7a",
    "Feminicídio Consumado": "#eda100",
    "Feminicídio Tentado": "#008300",
    "Lesão Corporal": "#4a3aa7",
}
MUTED = "#898781"
GRID = "#e1e0d9"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"


def style_ax(ax):
    """Estilo padrão do notebook: sem moldura, grade horizontal sutil."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK_SECONDARY, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linewidth=1)
    ax.set_axisbelow(True)


## 1. Tendência anual (2018-2025)

Para cada tipo de crime: regressão linear simples (ano → casos, nível estado) e
Mann-Kendall como checagem cruzada (mais robusto a não-linearidade). 2012-2017 fica
de fora (sem detalhe mensal, embora isso não importe aqui — é só para manter a janela
consistente com as outras seções) e 2026 fica de fora por ser ano parcial.


In [ ]:
tendencia_df = tendencia.main()
tendencia_df

In [ ]:
serie_anual = tendencia.carregar_serie_anual_estado()

fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
for ax, tipo in zip(axes, CRIME_ORDER):
    grupo = serie_anual[serie_anual.tipo_crime == tipo].sort_values("ano")
    linha = tendencia_df[tendencia_df.tipo_crime == tipo].iloc[0]
    cor = CRIME_COLORS[tipo]

    ax.scatter(grupo.ano, grupo.casos_total, color=cor, s=28, zorder=3)
    anos_linha = np.array([grupo.ano.min(), grupo.ano.max()])
    ax.plot(
        anos_linha,
        linha.slope * anos_linha + linha.intercept,
        color=cor,
        linewidth=2,
        alpha=0.55,
    )

    sig = "significativo" if linha.significativo_linregress_5pct else "não significativo"
    ax.set_title(f"{tipo}\np={linha.p_valor_linregress:.3f} · R²={linha.r2:.2f} · {sig}", fontsize=9)
    style_ax(ax)

fig.suptitle("Tendência anual por tipo de crime (2018-2025)", y=1.08)
fig.tight_layout()
plt.show()

**Leitura:** Ameaça, Estupro e Feminicídio Tentado têm tendência significativa a 5%
(queda para Ameaça e Feminicídio Tentado, alta para Estupro). Feminicídio Consumado e
Lesão Corporal não atingem significância — com n=8 anos, o poder estatístico é baixo.


## 2. Sazonalidade mensal (2018-2026)

Kruskal-Wallis entre os 12 meses do ano, por tipo de crime (cada grupo/mês é composto
pelas observações de todos os anos disponíveis). Onde significativo, os meses em
destaque vêm de um post-hoc Mann-Whitney U mês-vs-resto com correção de Holm-Bonferroni.


In [ ]:
kruskal_df, destaques_df = sazonalidade.main()
kruskal_df

In [ ]:
serie_mensal = sazonalidade.carregar_serie_mensal_estado()

fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
for ax, tipo in zip(axes, CRIME_ORDER):
    grupo = serie_mensal[serie_mensal.tipo_crime == tipo]
    medias = grupo.groupby("mes")["casos"].mean().reindex(range(1, 13))
    linha_kw = kruskal_df[kruskal_df.tipo_crime == tipo].iloc[0]
    cor = CRIME_COLORS[tipo]

    ax.bar(medias.index, medias.values, color=cor, width=0.65)

    if not destaques_df.empty:
        destaques_tipo = destaques_df[
            (destaques_df.tipo_crime == tipo) & (destaques_df.destaque_5pct)
        ]
        for _, d in destaques_tipo.iterrows():
            ax.text(
                d.mes,
                medias[d.mes] * 1.03,
                "*",
                ha="center",
                va="bottom",
                color=INK_PRIMARY,
                fontsize=14,
                fontweight="bold",
            )

    sig = "sig." if linha_kw.significativo_5pct else "não sig."
    ax.set_title(f"{tipo}\nKruskal-Wallis {sig} (p={linha_kw.p_valor:.1e})", fontsize=9)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(list(sazonalidade.MESES_NOMES.values()), rotation=45, ha="right", fontsize=7)
    style_ax(ax)

fig.suptitle(
    "Média de casos por mês, por tipo de crime (2018-2026) — * = mês em destaque (p ajustado < 0,05)",
    y=1.1,
)
fig.tight_layout()
plt.show()

**Leitura:** Ameaça, Estupro e Lesão Corporal têm diferença significativa entre meses.
Janeiro se destaca para cima nos três; junho (e maio/julho para Lesão Corporal) se
destaca para baixo. Feminicídio (tentado ou consumado) não mostra sazonalidade
detectável — esperado para eventos raros, com pouca contagem mensal.


## 3. Correlação entre tipos de crime, por município

Spearman entre os 5 tipos de crime, usando o total acumulado por município. Duas
variantes lado a lado: número absoluto de casos (2012-2026) e taxa por 100 mil
habitantes (2012-2025 — 2026 não tem estimativa de população ainda). A versão em taxa
controla o efeito óbvio de "município grande tem mais de tudo".


In [ ]:
resultados_corr = correlacao.main()
resultados_corr["casos_absolutos"][2]  # pares, do mais ao menos correlacionado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
fig.subplots_adjust(wspace=1.3)
titulos = {"casos_absolutos": "Número absoluto de casos", "taxa_por_100mil_hab": "Taxa por 100 mil hab."}

for ax, (nome, (matriz_corr, matriz_pvalor, pares)) in zip(axes, resultados_corr.items()):
    matriz_ordenada = matriz_corr.reindex(index=CRIME_ORDER, columns=CRIME_ORDER)
    im = ax.imshow(matriz_ordenada.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xticklabels(CRIME_ORDER, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CRIME_ORDER, fontsize=8)
    for i in range(5):
        for j in range(5):
            valor = matriz_ordenada.values[i, j]
            cor_texto = "white" if abs(valor) > 0.6 else INK_PRIMARY
            ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8, color=cor_texto)
    ax.set_title(titulos[nome], fontsize=10)
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.colorbar(im, ax=axes, shrink=0.8, label="rho de Spearman", pad=0.12)
fig.suptitle("Correlação de Spearman entre tipos de crime, por município")
plt.show()

**Leitura:** boa parte da correlação em números absolutos é só efeito de porte do
município — ao controlar por população, a maioria dos pares cai bastante (de ~0.7-0.9
para ~0.2-0.4). A exceção é **Ameaça × Lesão Corporal**, que continua forte mesmo
em taxa (0.82) — os dois parecem de fato andar juntos, e não só porque cidades grandes
têm mais casos de tudo.


## 4. Quebra estrutural em torno das enchentes de maio/2024

Teste de Chow sobre a série mensal dessazonalizada (resíduo em relação à média
histórica pré-enchente de cada mês do ano): compara um único ajuste linear
tempo→resíduo contra dois ajustes separados (pré e pós maio/2024).


In [ ]:
chow_df, mw_df = quebra_estrutural.main()
chow_df

In [ ]:
serie_mensal_qe = quebra_estrutural.carregar_serie_mensal_estado()
serie_dessaz = quebra_estrutural._dessazonalizar(serie_mensal_qe)

fig, axes = plt.subplots(1, 5, figsize=(24, 4.2))
for ax, tipo in zip(axes, CRIME_ORDER):
    grupo = serie_dessaz[serie_dessaz.tipo_crime == tipo].sort_values(["ano", "mes"]).reset_index(drop=True)
    t = grupo.index.to_numpy(dtype=float)
    y = grupo["residuo"].to_numpy()
    pos = (
        (grupo["ano"] > quebra_estrutural.ANO_ENCHENTE)
        | ((grupo["ano"] == quebra_estrutural.ANO_ENCHENTE) & (grupo["mes"] >= quebra_estrutural.MES_ENCHENTE))
    ).to_numpy()
    linha_chow = chow_df[chow_df.tipo_crime == tipo].iloc[0]
    cor = CRIME_COLORS[tipo]

    ax.plot(t, y, color=cor, linewidth=1, alpha=0.5)
    ax.axvline(t[pos][0], color=INK_SECONDARY, linestyle="--", linewidth=1)
    ax.plot(t[~pos], linha_chow.slope_pre * t[~pos] + linha_chow.intercepto_pre, color=cor, linewidth=2.5)
    ax.plot(
        t[pos],
        linha_chow.slope_pos * t[pos] + linha_chow.intercepto_pos,
        color=cor,
        linewidth=2.5,
        linestyle=(0, (4, 1)),
    )

    sig = "quebra sig." if linha_chow.significativo_5pct else "sem quebra sig."
    ax.set_title(f"{tipo}\nChow p={linha_chow.p_valor:.1e} ({sig})", fontsize=9)
    ax.set_xticks([])
    style_ax(ax)

fig.suptitle(
    "Série dessazonalizada (resíduo vs. média histórica pré-2024) — linha tracejada = maio/2024",
    y=1.1,
)
fig.tight_layout()
plt.show()

**Leitura:** Estupro tem a quebra mais nítida (p=1.6e-14) — a tendência de alta que
vinha desde 2018 se inverte após maio/2024. Feminicídio Tentado também mostra quebra
significativa (p=0.015). Ameaça fica no limiar (p=0.054, mudança de inclinação visível
mas não cruza o corte de 5%). Lesão Corporal e Feminicídio Consumado não mostram quebra
detectável neste desenho.

O post-hoc por mês equivalente (`mw_df`, Mann-Whitney U 2018-2023 vs. 2024/2025) não
detecta significância em nenhum mês/tipo — com n=6 vs n=2 por teste, o menor p-valor
exato possível é 0.143, então essa versão serve só como referência descritiva
(tamanho/direção do efeito), não como teste com poder estatístico real.
